# 06z_lr_promo0_logretention_only

Prepared notebook for log retention only model rerun. This notebook is generated for manual/team execution and was not executed during the prep goal.

## Scope

- Raw retention columns are forbidden.
- `log_retention_w2_ratio` and `log_retention_w3_ratio` must be features.
- `is_repurchase`, `USER_KEY`, `is_promotion`, and payment indicator columns are excluded from features.
- StratifiedKFold is used for comparability with existing PUBLIC notebooks. USER_KEY duplicate group leakage caveat remains.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd()
if not (ROOT / "PUBLIC").exists():
    ROOT = Path(r"C:\Code\ott-churn-prediction")
PUBLIC = ROOT / "PUBLIC"

DATA = ROOT / "PUBLIC/data/06z_model_input_promo_0_log_retention_only.csv"
OUT_DIR = ROOT / "PUBLIC/results/_06z_log_retention_only_model_rerun_260520/lr_promo0"
PROMO = 0
MODEL_NAME = "LogisticRegression baseline"
RANDOM_STATE = 42
N_TRIALS = 100
N_SPLITS = 5
OVERFIT_GAP = 0.03
TEST_SIZE = 0.2
GAP_PENALTY = 0.50

TARGET_COL = "is_repurchase"
ID_COL = "USER_KEY"
RAW_RETENTION_COLS = ["retention_w2_ratio", "retention_w3_ratio"]
LOG_RETENTION_COLS = ["log_retention_w2_ratio", "log_retention_w3_ratio"]
EXCLUDE_COLS = [
    TARGET_COL,
    ID_COL,
    "is_promotion",
    "payment_is_mobile",
    "payment_is_pc",
    "payment_is_android",
    "payment_is_ios",
] + RAW_RETENTION_COLS

OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)
if any(col in df.columns for col in RAW_RETENTION_COLS):
    raise ValueError("Raw retention columns must not exist in log retention only model input.")
missing_log = [col for col in LOG_RETENTION_COLS if col not in df.columns]
if missing_log:
    raise ValueError(f"Missing log retention columns: {missing_log}")
if TARGET_COL not in df.columns:
    raise ValueError("Missing target column is_repurchase.")

feature_cols = [col for col in df.columns if col not in EXCLUDE_COLS]
non_numeric_cols = df[feature_cols].select_dtypes(exclude="number").columns.tolist()
if non_numeric_cols:
    raise ValueError(f"Non-numeric feature columns found: {non_numeric_cols}")
if not set(LOG_RETENTION_COLS).issubset(feature_cols):
    raise ValueError("Log retention columns are not included as features.")

X = df[feature_cols].copy()
y = pd.to_numeric(df[TARGET_COL], errors="raise").astype(int)
missing_total = int(X.isna().sum().sum())
if missing_total:
    X = X.fillna(X.median(numeric_only=True))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

def make_model(params):
    return Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(**params))])

def objective(trial):
    params = {
        "C": trial.suggest_float("C", 0.001, 10.0, log=True),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "penalty": "l2",
        "solver": "lbfgs",
        "max_iter": 3000,
        "random_state": RANDOM_STATE,
    }

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    train_scores = []
    valid_scores = []
    for train_idx, valid_idx in cv.split(X_train, y_train):
        X_tr = X_train.iloc[train_idx]
        X_va = X_train.iloc[valid_idx]
        y_tr = y_train.iloc[train_idx]
        y_va = y_train.iloc[valid_idx]
        model = make_model(params)
        model.fit(X_tr, y_tr)
        train_pred = model.predict_proba(X_tr)[:, 1]
        valid_pred = model.predict_proba(X_va)[:, 1]
        train_scores.append(roc_auc_score(y_tr, train_pred))
        valid_scores.append(roc_auc_score(y_va, valid_pred))
    mean_train_auc = float(np.mean(train_scores))
    mean_valid_auc = float(np.mean(valid_scores))
    gap = mean_train_auc - mean_valid_auc
    objective_value = mean_valid_auc - GAP_PENALTY * max(0.0, gap)
    trial.set_user_attr("mean_train_auc", mean_train_auc)
    trial.set_user_attr("mean_valid_auc", mean_valid_auc)
    trial.set_user_attr("gap", gap)
    trial.set_user_attr("overfit", bool(gap > OVERFIT_GAP))
    trial.set_user_attr("objective_value", objective_value)
    return objective_value

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_TRIALS)

trial_rows = []
for trial in study.trials:
    row = {
        "trial": trial.number,
        "objective_value": trial.user_attrs.get("objective_value"),
        "mean_valid_auc": trial.user_attrs.get("mean_valid_auc"),
        "mean_train_auc": trial.user_attrs.get("mean_train_auc"),
        "gap": trial.user_attrs.get("gap"),
        "overfit": trial.user_attrs.get("overfit"),
    }
    for key, value in trial.params.items():
        row[f"param_{key}"] = value
    trial_rows.append(row)
trials = pd.DataFrame(trial_rows)
trials.to_csv(OUT_DIR / "trials_all.csv", index=False, encoding="utf-8-sig")

eligible = trials[trials["overfit"].eq(False)].copy()
if eligible.empty:
    selected = trials.sort_values(["objective_value", "mean_valid_auc"], ascending=False).iloc[0]
    selection_note = "WARN: no non-overfit trial found"
else:
    selected = eligible.sort_values(["objective_value", "mean_valid_auc"], ascending=False).iloc[0]
    selection_note = "PASS: selected best non-overfit trial"

best_trial = study.trials[int(selected["trial"])]
best_params = best_trial.params
model = make_model(best_params)
model.fit(X_train, y_train)
test_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)
train_proba = model.predict_proba(X_train)[:, 1]

final_train_auc = roc_auc_score(y_train, train_proba)
test_roc_auc = roc_auc_score(y_test, test_proba)
test_pr_auc = average_precision_score(y_test, test_proba)

final = {
    "model": MODEL_NAME,
    "promo": PROMO,
    "data_file": str(DATA),
    "n_rows": len(df),
    "n_features": len(feature_cols),
    "n_trials": N_TRIALS,
    "best_trial": int(selected["trial"]),
    "gap_penalty": GAP_PENALTY,
    "best_objective_value": float(selected["objective_value"]),
    "best_valid_auc": float(selected["mean_valid_auc"]),
    "best_train_auc": float(selected["mean_train_auc"]),
    "best_gap": float(selected["gap"]),
    "overfit": bool(selected["overfit"]),
    "test_roc_auc": float(test_roc_auc),
    "test_pr_auc": float(test_pr_auc),
    "test_f1": float(f1_score(y_test, test_pred)),
    "test_precision": float(precision_score(y_test, test_pred, zero_division=0)),
    "test_recall": float(recall_score(y_test, test_pred, zero_division=0)),
    "final_train_auc": float(final_train_auc),
    "final_gap_proxy": float(final_train_auc - test_roc_auc),
    "selection_note": selection_note,
    "cv_method": "StratifiedKFold",
    "group_leakage_caveat": "USER_KEY can be duplicated. StratifiedKFold is retained for comparability with PUBLIC notebooks; GroupKFold was not used in this prepared notebook.",
    "raw_retention_removed": True,
    "log_retention_used": True,
}
for key, value in best_params.items():
    final[f"param_{key}"] = value
pd.DataFrame([final]).to_csv(OUT_DIR / "final_result.csv", index=False, encoding="utf-8-sig")

feature_manifest = pd.DataFrame({
    "feature_name": feature_cols,
    "used_as_feature": True,
})
feature_manifest.to_csv(OUT_DIR / "feature_manifest_used.csv", index=False, encoding="utf-8-sig")

print(json.dumps(final, ensure_ascii=False, indent=2))


c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-05-20 14:53:11,038] A new study created in memory with name: no-name-f15f27cc-4ef7-467e-9bbb-67b9fe11c6b7
[I 2026-05-20 14:53:11,333] Trial 0 finished with value: 0.8595943526161787 and parameters: {'C': 0.03148911647956861, 'class_weight': None}. Best is trial 0 with value: 0.8595943526161787.
[I 2026-05-20 14:53:11,653] Trial 1 finished with value: 0.8602515470572853 and parameters: {'C': 0.24810409748678125, 'class_weight': None}. Best is trial 1 with value: 0.8602515470572853.
[I 2026-05-20 14:53:11,866] Trial 2 finished with value: 0.8496304884450074 and parameters: {'C': 0.0017073967431528124, 'class_weight': None}. Best is trial 1 with value: 0.8602515470572853.
[I 2026-05-20 14:53:12,203] 

{
  "model": "LogisticRegression baseline",
  "promo": 0,
  "data_file": "C:\\Code\\ott-churn-prediction\\PUBLIC\\data\\06z_model_input_promo_0_log_retention_only.csv",
  "n_rows": 11193,
  "n_features": 75,
  "n_trials": 100,
  "best_trial": 84,
  "gap_penalty": 0.5,
  "best_objective_value": 0.8611747188933192,
  "best_valid_auc": 0.8651377678675931,
  "best_train_auc": 0.873063865816141,
  "best_gap": 0.007926097948547928,
  "overfit": false,
  "test_roc_auc": 0.8604986173407958,
  "test_pr_auc": 0.9486434690282437,
  "test_f1": 0.8356867779204108,
  "test_precision": 0.9247159090909091,
  "test_recall": 0.7622950819672131,
  "final_train_auc": 0.872364190778083,
  "final_gap_proxy": 0.011865573437287225,
  "selection_note": "PASS: selected best non-overfit trial",
  "cv_method": "StratifiedKFold",
  "group_leakage_caveat": "USER_KEY can be duplicated. StratifiedKFold is retained for comparability with PUBLIC notebooks; GroupKFold was not used in this prepared notebook.",
  "raw_ret